# Notice utilisation 
Ce code permet d'effectuer la discrimination des assignations réalisées avec DataAnalysis 
Les colonnes que le fichier d'export des assignations doit contenir sont les suivantes : 
+ Observed Intens = intensité
+ Observed m/z = masse mesurée
+ err ppm = erreur sur la masse 
+ sum formula = formule brute proposée
+ et aussi une colonne pour chaque atome et chaque isotope

si les colonnes demandées n'existent pas alors le code remplira automatiquement avec des 0 : donc CONSERVER les colonnes initialement définies sur le code (C,H,N,O,S,Na,Cl,Mg et isotopes) même si il n'y a aucune assignation avec ces atomes car sinon le code va renvoyer des erreurs 

# Packages et fonctions

In [1257]:
import pandas as pd
import glob
import os
import numpy as np

# permet de faire les comparaisons masses-masses pour retirer les contaminations
def tronquer_4_decimales(x):
    try:
        x_str = f"{float(x):.10f}"
        partie_entiere, partie_decimale = x_str.split(".")
        return f"{partie_entiere}.{partie_decimale[:4]}"
    except:
        return None

def tronquer_3_decimales(x):
    try:
        x_str = f"{float(x):.10f}"
        partie_entiere, partie_decimale = x_str.split(".")
        return f"{partie_entiere}.{partie_decimale[:3]}"
    except:
        return None

## Fonctions pour attribuer l'indice de confiance

### Cas des molécules organiques classiques CHO, CHNO, CHNOS...

In [1261]:
# VALIDATION DU SOUFRE 
parametres_soufre = {
    1: {"norm": 95, "isotopes": {1: 4},  "lim": 7.1e7},
    2: {"norm": 90, "isotopes": {1: 8},  "lim": 3.4e7},
    3: {"norm": 86, "isotopes": {1: 11}, "lim": 2.4e7},
    4: {"norm": 82, "isotopes": {1: 14}, "lim": 1.8e7},
    5: {"norm": 77, "isotopes": {1: 17}, "lim": 1.4e7},
    6: {"norm": 74, "isotopes": {1: 20}, "lim": 1.2e7},
    7: {"norm": 70, "isotopes": {1: 22}, "lim": 9.6e6},}

def indice_confiance_CHNOS(df):
    df = df.copy()
    # Boucle sur chaque massif isotopique (groupe)
    group_cols = ['H','O','N','Na','C_tot','S_tot']
    results = []

    # Boucle sur chaque groupe unique de formule brute
    for _, group in df.groupby(group_cols):
        # Vérification de la présence de la formule mère (mono isotopique)
        mere = group[(group["^13C"] == 0) & (group["^34S"] == 0)]
        if mere.empty:
            continue
        intens_mere = mere["Observed Intens"].values[0]

        # Vérification de la présence des isotopes 
        has_13C = (group["^13C"] > 0).any()
        has_34S = (group["^34S"] > 0).any()
        Stot = group["S_tot"].iloc[0]

        if Stot > 0:  # CAS ATTRIBUTION AVEC SOUFRE 
            S_atoms = int(Stot)
            if S_atoms in parametres_soufre:
                val_centrale = parametres_soufre[S_atoms]["isotopes"][1]
                lim = parametres_soufre[S_atoms]["lim"]

                if has_34S: #^34S présent 
                    obs = group.loc[group["^34S"] > 0, "Observed Intens"].values
                    intens_obs = obs[0] 

                    intens_ref = intens_mere / parametres_soufre[S_atoms]["norm"]
                    borne_5  = (intens_ref * (val_centrale - 5), intens_ref * (val_centrale + 5))
                    borne_20 = (intens_ref * (val_centrale - 20), intens_ref * (val_centrale + 20))

                    if borne_5[0] <= intens_obs <= borne_5[1]:
                        idx = 1
                    elif borne_20[0] <= intens_obs <= borne_20[1]:
                        idx = 2
                    elif has_13C:
                        idx = 3
                    else:
                        continue
                else:  # ^34S absent
                    if has_13C:
                        idx = 3
                    elif intens_mere < lim:
                        idx = 4
                    else:
                        continue

        else:  # CAS ATTRIBUTION SANS SOUFRE 
            if has_13C:
                idx = 3
            else:
                idx = 4

        # Appliquer l’indice à toutes les lignes du groupe 
        group = group.copy()
        group["indice de confiance"] = idx
        results.append(group)

    # Concaténer tous les groupes valides
    if not results:
        return pd.DataFrame(columns=df.columns)

    attrib_valide_final = pd.concat(results).reset_index(drop=True)
    return attrib_valide_final

### Cas des molécules avec magnéisum

In [1263]:
# VALIDATION DU MAGNESIUM 
parametres_mg = {
    1: {"norm": 79, "isotopes": {1: 10, 2: 11}},  
}

def id_confiance_magnesium(row):
    isotopes = row['list_Mg']
    intensites = row['intensities']
    params = parametres_mg[1]

    i0 = isotopes.index(0)
    intens_M0 = intensites[i0]
    intens_ref = intens_M0 / params["norm"]

    # Vérification isotopes 25Mg et 26Mg
    check = {}
    for iso, val_centrale in params["isotopes"].items():
        if iso in isotopes:
            i_iso = isotopes.index(iso)
            intens_iso = intensites[i_iso]

            attendu = intens_ref * val_centrale
            borne_5 = (intens_ref * (val_centrale - 5),intens_ref * (val_centrale + 5))
            borne_20 = (intens_ref * (val_centrale - 20),intens_ref * (val_centrale + 20))

            if borne_5[0] <= intens_iso <= borne_5[1]:
                check[iso] = "5%"
            elif borne_20[0] <= intens_iso <= borne_20[1]:
                check[iso] = "20%"
            else:
                check[iso] = "out"
        else:
            check[iso] = "absent"

    # Attribution des niveaux de confiance
    if check[1] == "5%" and check[2] == "5%":
        return 1
    elif check[1] in ("5%", "20%") and check[2] in ("5%", "20%"):
        return 2
    elif check[1] in ("5%", "20%") or check[2] in ("5%", "20%"):
        return 3
    elif check[1] == "absent" and check[2] == "absent" and intens_M0 < 3e7:
        return 4
    else:
        return 0

def indice_confiance_Mg(df):
    df = df.copy()
    # Boucle sur chaque massif isotopique (groupe)
    group_cols = ['H','O','C_tot','Mg_tot','S_tot']
    results = []

    for _, group in df.groupby(group_cols):
        # Vérification de la présence de la formule mère (mono isotopique) 
        mere = group[
            (group['^13C'] == 0) &
            (group['^25Mg'] == 0) &
            (group['^26Mg'] == 0) &
            (group['^34S'] == 0)
        ]
        if mere.empty:
            continue # si pas de formule mère (ie. isotope seul) -> ne pas conserver 

        # Liste des isotopes et intensités
        mg_rows = group[(group['^13C']==0) & (group['^34S']==0)]
        list_Mg = []
        intensities = []
        for _, row in mg_rows.iterrows():
            if row['^25Mg'] > 0:
                iso = 1  # correspond à parametres_mg[1] -> ^25Mg
            elif row['^26Mg'] > 0:
                iso = 2  # correspond à parametres_mg[1] -> ^26Mg
            else:
                iso = 0  # mère 
            list_Mg.append(iso)
            intensities.append(row['Observed Intens'])

        # Calcul de l’indice de confiance
        indice = id_confiance_magnesium(pd.Series({
            'list_Mg': list_Mg,
            'intensities': intensities}))
        if indice == 0:
            continue  # pas de confiance → on rejette le groupe

        # Appliquer l’indice à toutes les lignes du groupe
        group = group.copy()
        group['indice de confiance'] = indice
        results.append(group)

    # Concaténer tous les groupes valides
    if not results:
        return pd.DataFrame(columns=df.columns)

    attrib_valide_final = pd.concat(results).reset_index(drop=True)

    return attrib_valide_final

### Cas des molécules avec chlore

In [1265]:
# VALIDATION DU CHLORE 
parametres_cl = {
    1: {"norm": 76, "isotopes": {1: 24}},
    2: {"norm": 57,  "isotopes": {1: 37, 2: 6}},
    3: {"norm": 44,  "isotopes": {1: 42, 2: 13}},
    4: {"norm": 33,  "isotopes": {1: 42, 2: 20, 3: 4}},
    5: {"norm": 25,  "isotopes": {1: 40, 2: 26, 3: 8}},
    6: {"norm": 19,  "isotopes": {1: 36, 2: 29, 3: 12}},
    7: {"norm": 14,  "isotopes": {1: 32, 2: 31, 3: 16}},
    8: {"norm": 11,  "isotopes": {1: 28, 2: 31, 3: 20}},
    9: {"norm": 8,  "isotopes": {1: 24, 2: 30, 3: 23}},
    10: {"norm": 6,  "isotopes": {1: 20, 2: 29, 3: 24}},
    11: {"norm": 5,  "isotopes": {1: 17, 2: 27, 3: 25}},
    12: {"norm": 4,  "isotopes": {1: 14, 2: 24, 3: 26}}}

def id_confiance_chlore(row):
    cl_tot = row['Cl_tot']
    isotopes = row['list_37Cl']
    intensites = row['intensities']

    params = parametres_cl[cl_tot]

    # Pic de référence 37Cl=0
    if 0 not in isotopes:
        return 0
    i0 = isotopes.index(0)
    intens_M0 = intensites[i0]
    intens_ref = intens_M0 / params["norm"]

    isotopes_valides = set()

    # Vérification des isotopes attendus
    for iso, val_centrale in params["isotopes"].items():
        if iso in isotopes:
            i_iso = isotopes.index(iso)
            intens_iso = intensites[i_iso]

            borne_min = intens_ref * (val_centrale - 5)
            borne_max = intens_ref * (val_centrale + 5)

            if borne_min <= intens_iso <= borne_max:
                isotopes_valides.add(iso)
    
    # Vérification des cas
    if cl_tot == 1:
        return 1 if 1 in isotopes_valides else 0

    elif cl_tot in [2, 3]:
        if {1, 2}.issubset(isotopes_valides):
            return 1
        elif 1 in isotopes_valides:
            return 2
        else:
            return 0

    elif 4 <= cl_tot <= 10:
        if {1, 2, 3}.issubset(isotopes_valides):
            return 1
        elif {1, 2}.issubset(isotopes_valides):
            return 2
        else:
            return 0
    return 0

def indice_confiance_Cl(df):
    df = df.copy()
    # Boucle sur chaque massif isotopique (groupe)
    group_cols = ['H','O','N','S_tot','C_tot','Cl_tot']
    results = []

    for _, group in df.groupby(group_cols):
        # Vérification de la présence de la formule mère (mono isotopique) 
        mere = group[
            (group['^13C'] == 0) &
            (group['^37Cl'] == 0) & 
            (group['^34S'] == 0)]
        if mere.empty:
            continue # si pas de formule mère (ie. isotope seul) -> ne pas conserver 

        # Liste des isotopes et intensités
        cl_rows = group[group['^37Cl'].isin([1,2,3]) | (group['^37Cl'] == 0)]
        list_37Cl = []
        intensities = []
        for _, row in cl_rows.iterrows():
            if row['^37Cl'] == 0 :
                iso = 0 # correspond à l'attribution mère 
            else :
                iso = row['^37Cl']  
            list_37Cl.append(iso)
            intensities.append(row['Observed Intens'])

        # Calcul de l’indice de confiance
        indice_cl = id_confiance_chlore(pd.Series({
            'Cl_tot': group['Cl_tot'].iloc[0],  # même pour tout le groupe
            'list_37Cl': list_37Cl,
            'intensities': intensities}))
        if indice_cl == 0:
            continue  # pas de confiance → rejet du groupe

        # Appliquer l’indice à toutes les lignes du groupe
        group = group.copy()
        group['indice de confiance'] = indice_cl
        results.append(group)

    # Concaténer tous les groupes valides
    if not results:
        return pd.DataFrame(columns=df.columns)

    attrib_valide_final = pd.concat(results).reset_index(drop=True)
    return attrib_valide_final

# 0. Importation des données

In [1267]:
#Import du dossier contenant les fichiers CSV des assignations 
dossier_csv = r"C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\ESI+\RAW\Murchison"

#Colonnes à conserver
colonnes_a_conserver = [
"Observed Intens",
"Observed m/z",
"err ppm",
"sum formula",
'C',
'H',
'N',
'O',
'S',
'Mg',
'Cl',
'Na',
'^13C',
'^34S',
'^25Mg',
'^26Mg',
'^37Cl',
]

# Fusionner tous les fichiers du dossier (dans le cas ou les assignations ont été réalisées par passes et qu'il y a un fichier par famille) 
fichiers = glob.glob(os.path.join(dossier_csv, "*.csv"))
liste_df = [pd.read_csv(fichier, sep=";") for fichier in fichiers]
df_final = pd.concat(liste_df, ignore_index=True, sort=False)

# Ajout automatique des colonnes manquantes
for col in colonnes_a_conserver:
    if col not in df_final.columns:
        df_final[col] = 0

#Colonne à conserver
df_final = df_final[colonnes_a_conserver]
df_final = df_final.fillna(0)

# Calcul du DBE et des totaux avec isotopes 
df_final["C_tot"]  = df_final["C"] + df_final["^13C"]
df_final["S_tot"]  = df_final["S"] + df_final["^34S"]
df_final["Mg_tot"] = df_final["Mg"] + df_final["^25Mg"] + df_final["^26Mg"]
df_final["Cl_tot"] = df_final["Cl"] + df_final["^37Cl"]
df_final["DBE"] = 1+ df_final["C_tot"]+ (df_final["N"]/2) - (df_final["H"]/2) - (df_final["Cl_tot"] / 2)

df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,^13C,^34S,^25Mg,^26Mg,^37Cl,C_tot,S_tot,Mg_tot,Cl_tot,DBE
0,31468532,153.138632,-0.047,C9 H17 N2,9,17,2.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0,9,0.0,0.0,0,2.5
1,7415171,155.154288,-0.081,C9 H19 N2,9,19,2.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0,9,0.0,0.0,0,1.5
2,9434708,156.174680,-0.027,C10 H22 N,10,22,1.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0,10,0.0,0.0,0,0.5
3,5200452,158.096428,-0.012,C11 H12 N,11,12,1.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0,11,0.0,0.0,0,6.5
4,14536860,161.107335,-0.060,C10 H13 N2,10,13,2.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0,10,0.0,0.0,0,5.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28945,16995822,792.502009,-0.141,C43 H79 Mg O4 S3 ^13C,43,79,0.0,4.0,3.0,1.0,...,1,0.0,0.0,0.0,0,44,3.0,1.0,0,5.5
28946,16995822,792.502009,-0.213,C40 H79 O9 S2 ^25Mg,40,79,0.0,9.0,2.0,0.0,...,0,0.0,1.0,0.0,0,40,2.0,1.0,0,1.5
28947,7745956,795.544008,-0.009,C44 H86 Mg O S4 ^13C,44,86,0.0,1.0,4.0,1.0,...,1,0.0,0.0,0.0,0,45,4.0,1.0,0,3.0
28948,7745956,795.544008,0.279,C47 H79 Mg O6 S,47,79,0.0,6.0,1.0,1.0,...,0,0.0,0.0,0.0,0,47,1.0,1.0,0,8.5


# 1. Filtrage des données

In [1269]:
# Calcul des ratios
df_final["H/C"] = df_final["H"] / df_final["C_tot"]
df_final["N/C"] = df_final["N"] / df_final["C_tot"]
df_final["O/C"] = df_final["O"] / df_final["C_tot"]
df_final["S/C"] = df_final["S_tot"] / df_final["C_tot"]

# Vérification du DBE (règle de l'azote)
# Cas 1 : Ionisation [M+Na]+
ionisation_na = (
    (df_final["Na"] == 1)
    & (df_final["DBE"] >= 0)
    & (df_final["DBE"] % 1 == 0)) # DBE entier

# Cas 2 : Ionisation [M+H]+
ionisation_h = (
    (df_final["Na"] == 0)
    & (df_final["DBE"] >= -0.5)
    & (df_final["DBE"] % 1 == 0.5)) #DBE demi entier 

# Filtre global : appliquer les fitres de ratio souhaités 
filtre = (
    (df_final["err ppm"].between(-0.2, 0.2)) 
    & (df_final["Observed m/z"].between(150, 800))
    & (df_final["H/C"].between(0.5, 2.5)) 
    & (df_final["N/C"] <= 1)
    & (df_final["O/C"] <= 1)
    & (df_final["S/C"] <= 0.8)
    & (ionisation_na | ionisation_h)
)

df_final = df_final[filtre].copy()
df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,^37Cl,C_tot,S_tot,Mg_tot,Cl_tot,DBE,H/C,N/C,O/C,S/C
0,31468532,153.138632,-0.047,C9 H17 N2,9,17,2.0,0.0,0.0,0.0,...,0,9,0.0,0.0,0,2.5,1.888889,0.222222,0.000000,0.000000
1,7415171,155.154288,-0.081,C9 H19 N2,9,19,2.0,0.0,0.0,0.0,...,0,9,0.0,0.0,0,1.5,2.111111,0.222222,0.000000,0.000000
2,9434708,156.174680,-0.027,C10 H22 N,10,22,1.0,0.0,0.0,0.0,...,0,10,0.0,0.0,0,0.5,2.200000,0.100000,0.000000,0.000000
3,5200452,158.096428,-0.012,C11 H12 N,11,12,1.0,0.0,0.0,0.0,...,0,11,0.0,0.0,0,6.5,1.090909,0.090909,0.000000,0.000000
4,14536860,161.107335,-0.060,C10 H13 N2,10,13,2.0,0.0,0.0,0.0,...,0,10,0.0,0.0,0,5.5,1.300000,0.200000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28924,4943523,763.523394,0.180,C41 H85 Mg O2 S3 ^34S,41,85,0.0,2.0,3.0,1.0,...,0,41,4.0,1.0,0,-0.5,2.073171,0.000000,0.048780,0.097561
28929,6274015,765.533513,0.199,C46 H77 Mg O5 S,46,77,0.0,5.0,1.0,1.0,...,0,46,1.0,1.0,0,8.5,1.673913,0.000000,0.108696,0.021739
28943,4806820,790.588612,0.063,C52 H81 Mg O S ^13C,52,81,0.0,1.0,1.0,1.0,...,0,53,1.0,1.0,0,13.5,1.528302,0.000000,0.018868,0.018868
28944,31151962,791.498664,-0.154,C44 H79 Mg O4 S3,44,79,0.0,4.0,3.0,1.0,...,0,44,3.0,1.0,0,5.5,1.795455,0.000000,0.090909,0.068182


# 2. Recherche et suppression des contaminants

### Suppression des contaminants de la liste de masse totale (issue du peak picking) 
Permettra à la fin de la procédure de post-traitement de pouvoir comparer la liste des points assignés à la liste de masse totale et en déduire le nombre de pics non assignés 

In [1273]:
# fichier excel contenant la liste de masse brute issue du peak picking 
fichier_spectre = r'C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\ESI+\Murchison.xlsx'
mass_liste_brute = pd.read_excel(fichier_spectre,sheet_name='mass_list_brute')  
# fichier UWPR contenant la liste des contaminants usuels (Keller+ 2008) 
df_contaminants = pd.read_excel(r'C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\UWPR_CommonMassSpecContaminants.xls',sheet_name='Positive')  
# fichier contenant la liste des pics intenses du blanc (I > 1%) 
df_blanc = pd.read_excel(r'C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\ESI+\BLANC\BLANC.xlsx')

# Contaminants usuels (recherche à 4 décimales)

df_contaminants["masse_tronquee"] = (df_contaminants["Mass"].apply(tronquer_4_decimales))
mass_liste_brute["masse_tronquee_4"] = (mass_liste_brute["m/z"].apply(tronquer_4_decimales))
masses_contaminants = (set(df_contaminants["masse_tronquee"])&set(mass_liste_brute["masse_tronquee_4"]))
liste_contaminants_usuels = (df_contaminants[df_contaminants["masse_tronquee"].isin(masses_contaminants)][["Mass","Ion type","Formula for M or subunit or sequence","Possible origin and other comments"]])


# Contaminants du blanc d'extraction (recherche à 3 décimales)

df_blanc["masse_tronquee"] = (df_blanc["m/z"].apply(tronquer_3_decimales))
mass_liste_brute["masse_tronquee_3"] = (mass_liste_brute["m/z"].apply(tronquer_3_decimales))
masses_blanc = (set(df_blanc["masse_tronquee"])&set(mass_liste_brute["masse_tronquee_3"]))
liste_blanc = (mass_liste_brute[mass_liste_brute["masse_tronquee_3"].isin(masses_blanc)][["m/z", "I"]])

# Suppression des contaminants trouvés 
liste_masse_filtrée = mass_liste_brute[
    (~mass_liste_brute["masse_tronquee_4"].isin(masses_contaminants))
    &
    (~mass_liste_brute["masse_tronquee_3"].isin(masses_blanc))
].copy()

# Suppression des colonnes inutiles 
colonnes_a_supprimer = ["masse_tronquee_4","masse_tronquee_3"]
liste_masse_filtrée.drop(columns=[c for c in colonnes_a_supprimer if c in liste_masse_filtrée.columns],inplace=True)

# Ajout des feuilles dans le fichier excel

with pd.ExcelWriter(
    fichier_spectre,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    liste_contaminants_usuels.to_excel(
        writer,
        sheet_name="Liste_contaminants_usuels",
        index=False)

    liste_blanc.to_excel(
        writer,
        sheet_name="Liste_contaminants_blanc",
        index=False)

    liste_masse_filtrée.to_excel(
        writer,
        sheet_name="Liste_masse_finale",
        index=False)

print("Traitement terminé.")

Traitement terminé.


### Suppression des contaminants de la liste des assignations 

In [1275]:
# On conserve uniquement les masses qui sont présentent dans la liste de masse finale (sans les contaminants) 
masses_conservees = set(liste_masse_filtrée["m/z"].round(5)) # la liste de masse issue du peak picking contient 5 décimales, il faut arrondir
df_final = df_final[df_final["Observed m/z"].round(5).isin(masses_conservees)].copy()

df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,^37Cl,C_tot,S_tot,Mg_tot,Cl_tot,DBE,H/C,N/C,O/C,S/C
1,7415171,155.154288,-0.081,C9 H19 N2,9,19,2.0,0.0,0.0,0.0,...,0,9,0.0,0.0,0,1.5,2.111111,0.222222,0.000000,0.000000
2,9434708,156.174680,-0.027,C10 H22 N,10,22,1.0,0.0,0.0,0.0,...,0,10,0.0,0.0,0,0.5,2.200000,0.100000,0.000000,0.000000
3,5200452,158.096428,-0.012,C11 H12 N,11,12,1.0,0.0,0.0,0.0,...,0,11,0.0,0.0,0,6.5,1.090909,0.090909,0.000000,0.000000
5,11934135,165.138638,-0.082,C10 H17 N2,10,17,2.0,0.0,0.0,0.0,...,0,10,0.0,0.0,0,3.5,1.700000,0.200000,0.000000,0.000000
6,8185808,166.122636,0.030,C10 H16 N O,10,16,1.0,1.0,0.0,0.0,...,0,10,0.0,0.0,0,3.5,1.600000,0.100000,0.100000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28924,4943523,763.523394,0.180,C41 H85 Mg O2 S3 ^34S,41,85,0.0,2.0,3.0,1.0,...,0,41,4.0,1.0,0,-0.5,2.073171,0.000000,0.048780,0.097561
28929,6274015,765.533513,0.199,C46 H77 Mg O5 S,46,77,0.0,5.0,1.0,1.0,...,0,46,1.0,1.0,0,8.5,1.673913,0.000000,0.108696,0.021739
28943,4806820,790.588612,0.063,C52 H81 Mg O S ^13C,52,81,0.0,1.0,1.0,1.0,...,0,53,1.0,1.0,0,13.5,1.528302,0.000000,0.018868,0.018868
28944,31151962,791.498664,-0.154,C44 H79 Mg O4 S3,44,79,0.0,4.0,3.0,1.0,...,0,44,3.0,1.0,0,5.5,1.795455,0.000000,0.090909,0.068182


# 3. Discrimination des assignations

## Attribution d'un indice de confiance

In [1277]:
# Séparer les assignations en 3 catégories pour attribuer un indice de confiance en fonction des critères de chaque catégories : chlore, magnésium ou classique
def def_categorie(df):
    df = df.copy()
    df["categorie"] = np.where(df["Mg_tot"] > 0,"Mg",np.where(df["Cl_tot"] > 0, "Cl", "CHNOS"))
    return df

def séparer_categories(df):
    return {
        "CHNOS": df[df["categorie"] == "CHNOS"].copy(),
        "Mg": df[df["categorie"] == "Mg"].copy(),
        "Cl": df[df["categorie"] == "Cl"].copy()
    }

def attribution_indice_confiance(df):

    df = def_categorie(df)
    petit_df = séparer_categories(df)

    df_chnos = indice_confiance_CHNOS(petit_df["CHNOS"])
    df_mg    = indice_confiance_Mg(petit_df["Mg"])
    df_cl    = indice_confiance_Cl(petit_df["Cl"])

    df_fusion = pd.concat([df_chnos, df_mg, df_cl], ignore_index=True)

    colonnes_a_supprimer = ["categorie"]

    df_final = df_fusion.drop(columns=[c for c in colonnes_a_supprimer if c in df_fusion.columns])

    return df_final

df_final = attribution_indice_confiance(df_final)

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_13052\3347299230.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_fusion = pd.concat([df_chnos, df_mg, df_cl], ignore_index=True)


In [1278]:
# création d'un identifiant de famille pour récupérer tous les pics d'un massif isotopique
df_final["famille"] = (
    df_final["C_tot"].astype(str) + "_" +
    df_final["H"].astype(str)     + "_" +
    df_final["O"].astype(str)     + "_" +
    df_final["N"].astype(str)     + "_" +
    df_final["S_tot"].astype(str) + "_" +
    df_final["Mg_tot"].astype(str)+ "_" +
    df_final["Cl_tot"].astype(str)+"_"+
    df_final["Na"].astype(str)
)

## A) Utilisation des indices de confiance

In [1279]:
# Premier filtrage 
attributions_validees = pd.DataFrame(columns=df_final.columns) # initialisation de la liste des assignations validées
attributions_multiples = pd.DataFrame(columns=df_final.columns) # initialisation de la liste des mutli-assignations
masses_ignorees = set() # création liste des masses ignorées une fois traitées 

for masse in sorted(df_final["Observed m/z"].unique()): # parcours la liste de masse de manière croissante
    if masse in masses_ignorees : 
        continue 

    # Première étape : pour chaque masse on conserve UNIQUEMENT les points avec le meilleur indice de confiance (le plus petit) 
    df_masse = df_final[df_final["Observed m/z"] == masse] # récupération de toutes les assignations pour une masse donnée
    best_confiance = df_masse["indice de confiance"].min() # parmi les assignations proposées choisir le meilleur indice de confiance (le plus petit)
    liste_best = df_masse[df_masse["indice de confiance"] == best_confiance] # conserver uniquement les assignations avec le meilleur indice de confiance

    if len(liste_best) == 1:
        # une seule assignation est proposée au meilleur niveau de confiance = assigation validée
        selected = liste_best.iloc[0]
        famille = selected["famille"]
        famille_rows = df_final[
            (df_final["famille"] == famille) &
            (~df_final["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_validees = pd.concat([attributions_validees, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses validées de la liste de masse à traiter 
    else:
        # plusieurs assignations sont proposées au meilleur niveau de confiance = multi-assignation
        familles = liste_best["famille"].unique()
        famille_rows = df_final[
            (df_final["famille"].isin(familles)) &
            (~df_final["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_multiples = pd.concat([attributions_multiples, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses de la liste de masse à traiter 

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_13052\3715159674.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_validees = pd.concat([attributions_validees, famille_rows])
C:\Users\arthozoc\AppData\Local\Temp\ipykernel_13052\3715159674.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_multiples = pd.concat([attributions_multiples, famille_rows])


## B) Restriction des ratios atomiques 

In [1208]:
# Filtrage des multi_attributions pour ne conserver que celles avec un ratio plus probable
attributions_multiples["ALL/C"] = (attributions_multiples["N"]+attributions_multiples["O"]+attributions_multiples["S_tot"]) / attributions_multiples["C_tot"]

filtre = (
    ((attributions_multiples["N/C"] <= 0.4)
    & (attributions_multiples["O/C"] <= 0.8)
    & (attributions_multiples["S/C"] <= 0.4)
    & (attributions_multiples["ALL/C"] <= 1)))

attributions_multiples = attributions_multiples[filtre].copy()
colonnes_a_supprimer = ["ALL/C"]
attributions_multiples = attributions_multiples.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_multiples.columns])

In [1209]:
attributions_multiples_B = pd.DataFrame(columns=attributions_multiples.columns) # nouvelle liste des attributions multiples à l'issue de ce filtrage 
masses_ignorees = set()

# Parcourir de nouveau la liste de masse et ne conserver que si une seule proposition par masse 
for masse in sorted(attributions_multiples["Observed m/z"].unique()): # parcours la liste de masse de manière croissante
    if masse in masses_ignorees : 
        continue
        
    # récupération de toutes les assignations pour une masse donnée
    df_masse = attributions_multiples[attributions_multiples["Observed m/z"] == masse] 
   
    if len(df_masse) == 1:
        # une seule assignation est proposée après filtrage = assignation validée
        selected = df_masse.iloc[0]
        famille = selected["famille"]
        famille_rows = attributions_multiples[
            (attributions_multiples["famille"] == famille) &
            (~attributions_multiples["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_validees = pd.concat([attributions_validees, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses validées de la liste de masse à traiter 
    else:
        # plusieurs assignations sont proposées après filtrage = multi-assignations
        familles = df_masse["famille"].unique()
        famille_rows = attributions_multiples[
            (attributions_multiples["famille"].isin(familles)) &
            (~attributions_multiples["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_multiples_B = pd.concat([attributions_multiples_B, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses de la liste de masse à traiter 

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_13052\3122485576.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_multiples_B = pd.concat([attributions_multiples_B, famille_rows])


## C) Recherche des familles en CH2

In [1211]:
# Construire un identifiant famille_ch2 pour les 2 listes 
attributions_multiples_B["famille_ch2"] = (
    attributions_multiples_B["N"].astype(str) + "_" +
    attributions_multiples_B["O"].astype(str) + "_" +
    attributions_multiples_B["S_tot"].astype(str) + "_" +
    attributions_multiples_B["Mg_tot"].astype(str) + "_" +
    attributions_multiples_B["Na"].astype(str) + "_" +
    attributions_multiples_B["Cl_tot"].astype(str) + "_" +
    attributions_multiples_B["DBE"].astype(str))

attributions_validees["famille_ch2"] = (
    attributions_validees["N"].astype(str) + "_" +
    attributions_validees["O"].astype(str) + "_" +
    attributions_validees["S_tot"].astype(str) + "_" +
    attributions_validees["Mg_tot"].astype(str) + "_" +
    attributions_validees["Na"].astype(str) + "_" +
    attributions_validees["Cl_tot"].astype(str) + "_" +
    attributions_validees["DBE"].astype(str))

# Chercher pour chaque attribution de la liste attributions_multiples_B si sa famille en CH2 est déjà présente dans attributions_validees 
familles_CH2_attributions_validees = set(attributions_validees["famille_ch2"])

attributions_multiples_B["famille_CH2_validee"] = (attributions_multiples_B["famille_ch2"].isin(familles_CH2_attributions_validees).map({True: "OUI", False: "NON"}))

In [1212]:
attributions_multiples_C = pd.DataFrame(columns=attributions_multiples_B.columns) # nouvelle liste des attributions multiples à l'issue de ce filtrage 
masses_ignorees = set()

# Parcourir de nouveau la liste de masse et ne conserver que si une seule proposition par masse 
for masse in sorted(attributions_multiples_B["Observed m/z"].unique()): # parcours la liste de masse de manière croissante
    if masse in masses_ignorees : 
        continue 
    # récupération de toutes les assignations pour une masse donnée
    df_masse = attributions_multiples_B[attributions_multiples_B["Observed m/z"] == masse] 
    # combien d'assignation ont déjà leur famille CH2 validée 
    df_oui = df_masse[df_masse["famille_CH2_validee"]=="OUI"]
    if len(df_oui) == 1:
        # une seule assignation a une famille en CH2 deja validée = assignation validée
        selected = df_oui.iloc[0]
        famille = selected["famille"]
        famille_rows = attributions_multiples_B[
            (attributions_multiples_B["famille"] == famille) &
            (~attributions_multiples_B["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_validees = pd.concat([attributions_validees, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses validées de la liste de masse à traiter 
    else:
        # plusieurs assignations ont une famille en CH2 deja validée = multi-assignations
        familles = df_masse["famille"].unique()
        famille_rows = attributions_multiples_B[
            (attributions_multiples_B["famille"].isin(familles)) &
            (~attributions_multiples_B["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_multiples_C = pd.concat([attributions_multiples_C, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses de la liste de masse à traiter 

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_13052\342984146.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_multiples_C = pd.concat([attributions_multiples_C, famille_rows])


## D) Sauvegarde des deux listes : multi-attributions et attributions validées + tableau récap 

In [1214]:
colonnes_a_supprimer = ["C_tot","S_tot","Cl_tot","Mg_tot","indice de confiance", "famille", "famille_ch2","famille_CH2_validee"] # choix des colonnes à ne pas enregistrer 
attributions_multiples_finales = attributions_multiples_C.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_multiples_C.columns])
attributions_validees_finales = attributions_validees.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_validees.columns])

In [1215]:
nb_total = len(liste_masse_filtrée) # liste de masse totale issue du peak picking (sans les contaminants)
# calcul du nombre de pics attribués - multiattribués - non attribués 
nb_attribuees = attributions_validees_finales["Observed m/z"].nunique()
nb_multi = attributions_multiples_finales["Observed m/z"].nunique()
nb_non_attribuees = nb_total - nb_attribuees - nb_multi
# calcul du pourcentage de ces pics 
pourcentage_attribuees = (nb_attribuees * 100) / nb_total
pourcentage_multi = (nb_multi * 100) / nb_total
pourcentage_non_attribuees = (nb_non_attribuees * 100) / nb_total

df_resume = pd.DataFrame({
    "Catégorie": [
        "Masses attribuées",
        "Masses multi-attribuées",
        "Masses non attribuées"],
    "Nombre": [
        nb_attribuees,
        nb_multi,
        nb_non_attribuees],
    "% du total": [
        pourcentage_attribuees,
        pourcentage_multi,
        pourcentage_non_attribuees]})

with pd.ExcelWriter(
    fichier_spectre,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    attributions_validees_finales.to_excel(
        writer,
        sheet_name="Attributions_validées",
        index=False)

    attributions_multiples_finales.to_excel(
        writer,
        sheet_name="Multi_attributions",
        index=False)

    df_resume.to_excel(
        writer,
        sheet_name="Résumé",
        index=False)

# 4. Passage en formules moléculaires 

## A) Massif isotopique - conserver uniquement les molécules mères

In [1218]:
def construire_formules_meres(df):
    # Intensité totale de chaque famille (mère + isotopes)
    intensites = (df.groupby("famille")["Observed Intens"].sum().rename("Intensite_totale"))

    # Conserver uniquement les formules mères
    meres = df[
        (df["^13C"] == 0) &
        (df["^34S"] == 0) &
        (df["^25Mg"] == 0) &
        (df["^26Mg"] == 0) &
        (df["^37Cl"] == 0)
    ].copy()

    # Ajouter une nouvelle colonne intensité totale
    meres = meres.merge(
        intensites,
        left_on="famille",
        right_index=True,
        how="left")

    return meres

attributions_validees_meres = construire_formules_meres(attributions_validees)
attributions_multiples_meres = construire_formules_meres(attributions_multiples_C)

In [1219]:
colonnes_a_supprimer = ["C_tot","S_tot","Cl_tot","Mg_tot","^13C","^25Mg","^26Mg","^34S","^37Cl","indice de confiance", "famille", "famille_ch2","famille_CH2_validee"] # choix des colonnes à ne pas enregistrer 
attributions_multiples_meres = attributions_multiples_meres.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_multiples_meres.columns])
attributions_validees_meres = attributions_validees_meres.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_validees_meres.columns])

## B) Adduits d'ionisation - passage des formules ioniques aux formules moléculaires

In [1221]:
def trouver_adduit(df):
    masque_na = df["Na"] > 0
    # Adduits Na+
    df.loc[masque_na, "Na"] = (df.loc[masque_na, "Na"] - 1)
    # Adduits H+
    df.loc[~masque_na, "H"] = (df.loc[~masque_na, "H"] - 1) # retirer atome H
    df.loc[~masque_na, "DBE"] = (df.loc[~masque_na, "DBE"] + 0.5) # retirer 0,5 au DBE 

    return df

In [1222]:
validees_adduits = attributions_validees_meres.copy()
multi_adduits = attributions_multiples_meres.copy()

validees_adduits = trouver_adduit(attributions_validees_meres)
multi_adduits = trouver_adduit(attributions_multiples_meres)
validees_adduits = validees_adduits.drop(columns="Na")
multi_adduits = multi_adduits.drop(columns="Na")

## C) Sauvegarde des deux listes : multi-attributions et attributions validées : formules moléculaires

In [1224]:
with pd.ExcelWriter(
    fichier_spectre,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    validees_adduits.to_excel(
        writer,
        sheet_name="Formules_moléculaires_validées",
        index=False)

    multi_adduits.to_excel(
        writer,
        sheet_name="Formules_moléculaires_multi",
        index=False)

In [1225]:
# petit klaxon pour annoncer que le traitement est terminé
from IPython.display import Audio

samples = np.sin(2 * np.pi * 440 * np.linspace(0, 1, 44100))
display(Audio(samples, rate=44100, autoplay=True))